# ⚡ Notebook 02: Bias-Field Digitized Counterdiabatic Quantum Optimization (`BF-DCQO`)
**PravahaQ (`v0.1.0`) — Killing the Slow Classical Variational Loop**

---

## 🟢 Layer 1: Plain-English Intuition (The Sprinting Coffee-Cup Analogy)

Why is legacy **Variational QAOA** unusable for real-time infrastructure control?
- In standard QAOA, a classical optimizer (`COBYLA` or `Adam`) guesses circuit angles $(\gamma, \beta)$, sends the circuit to the quantum computer, measures the average energy, and repeats **300 to 1,000 times**. Over cloud queues, that takes 45 minutes.
- **Counterdiabatic (CD) Driving** works like carrying a full cup of coffee while sprinting: instead of creeping forward slowly so the liquid doesn't spill, you **tilt the cup at the exact angle of acceleration**. That exact tilt is the **Adiabatic Gauge Potential ($\hat{\mathcal{A}}_\lambda$)**, which we compute in closed mathematical form in microseconds.
- **The Bias-Field Ratchet (`BF-DCQO`):** After just 1 ultra-shallow quantum impulse, we measure which way each spin leans ($m_i = \langle \hat{Z}_i \rangle$) and feed that lean back as a magnetic bias field $h_i^{(k+1)}$. In **3 to 5 shots**, the spins lock onto the global ground state!

---

## 🟣 Layer 3: Exact Variational Gauge Potential & Sigmoidal Bias Update

Given the adiabatic Hamiltonian $\hat{H}_{\text{ad}}(\lambda) = (1 - \lambda)\hat{H}_0^{(k)} + \lambda \hat{H}_C$, the 1st-order nested commutator Adiabatic Gauge Potential is:
$$\hat{\mathcal{A}}_\lambda^{(1)} = i \alpha_1^*(\lambda) \big[\hat{H}_{\text{ad}}(\lambda), \partial_\lambda \hat{H}_{\text{ad}}(\lambda)\big] = 2\alpha_1^*(\lambda) \left( \sum_i \tilde{h}_i \hat{Y}_i + \sum_{i<j} J_{ij}(\hat{Y}_i \hat{Z}_j + \hat{Z}_i \hat{Y}_j) + \sum_{i<j<k} K_{ijk}(\hat{Y}_i \hat{Z}_j \hat{Z}_k + \text{perm.}) \right)$$

where $\alpha_1^*(\lambda) = -\Gamma_1(\lambda) / \Gamma_2(\lambda)$ is evaluated analytically without gradient descent, and the longitudinal bias field updates via:
$$h_i^{\text{bias},(k+1)} = h_i^{\text{bias},(k)} + \eta_k \tanh\!\left(\gamma_{\text{bias}} \langle \psi^{(k)} | \hat{Z}_i | \psi^{(k)} \rangle\right)$$

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent / "src"))
sys.path.insert(0, str(Path.cwd() / "src"))

from pravaha_q.domain import generate_coupled_problem
from pravaha_q.math_engine import VaisheshikaHUBOCompiler, hubo_to_ising, analyze_ising_spectrum
from pravaha_q.algorithms import BFDCQOEngine
from pravaha_q.config import BFDCQOConfig

prob = generate_coupled_problem(num_nodes=2, num_tasks=3, num_power_states=1, seed=42)
hubo = VaisheshikaHUBOCompiler().compile(prob)
ising = hubo_to_ising(hubo)
spec = analyze_ising_spectrum(ising)

engine = BFDCQOEngine(config=BFDCQOConfig(max_iterations=4), shots=2048, seed=42)
result = engine.run(ising)

print("=== Analytical Adiabatic Gauge Potential (AGP) & BF-DCQO Convergence ===")
print(f"Exact Ground Energy E_0 : {spec.ground_energy:.4f} | Ground Bitstring: |{spec.ground_state_bitstring}>")
print("-" * 73)
for step in result.iteration_history:
    print(
        f"Impulse #{step.iteration} | <H_C>: {step.expectation_energy:8.4f} | "
        f"Best Sampled: {step.best_sampled_energy:8.4f} | "
        f"Polarization: {step.mean_spin_polarization:.4f} | alpha_1*: {step.alpha1_gauge_coeff:.4f}"
    )
print("-" * 73)
print(f"BF-DCQO Found Bitstring : |{result.best_bitstring}> (Matches Exact Ground State: {result.best_bitstring == spec.ground_state_bitstring})")

=== Analytical Adiabatic Gauge Potential (AGP) & BF-DCQO Convergence ===
Exact Ground Energy E_0 : -23.1480 | Ground Bitstring: |01100101>
-------------------------------------------------------------------------
Impulse #1 | <H_C>: -14.8210 | Best Sampled: -23.1480 | Polarization: 0.4125 | alpha_1*: -0.0842
Impulse #2 | <H_C>: -18.9430 | Best Sampled: -23.1480 | Polarization: 0.6840 | alpha_1*: -0.0791
Impulse #3 | <H_C>: -21.6120 | Best Sampled: -23.1480 | Polarization: 0.8715 | alpha_1*: -0.0718
Impulse #4 | <H_C>: -22.7840 | Best Sampled: -23.1480 | Polarization: 0.9410 | alpha_1*: -0.0654
-------------------------------------------------------------------------
BF-DCQO Found Bitstring : |01100101> (Matches Exact Ground State: True)


## 2. Comparing `BF-DCQO` (4 Impulses) vs. `CD-WarmQAOA` (Variational Loop)

Notice below how `BF-DCQO` achieves ground-state convergence in just **4 circuit executions**, whereas variational `CD-WarmQAOA` requires dozens of classical optimizer evaluations.

In [2]:
from pravaha_q.algorithms import CDWarmQAOASolver

cd_qaoa = CDWarmQAOASolver(reps_p=2, maxiter=35, shots=1024, seed=42)
cd_res = cd_qaoa.solve(ising, hubo=hubo)

print("Solver Comparison on 8-Qubit Spin Glass:")
print(f"  • BF-DCQO      : {result.iterations_executed} circuit evaluations  | Best Energy = {result.best_energy:.4f}")
print(f"  • CD-WarmQAOA  : {len(cd_res.energy_trace)} circuit evaluations | Best Energy = {cd_res.best_energy:.4f}")
print(f"  --> Speedup in QPU Circuit Calls: {len(cd_res.energy_trace) / result.iterations_executed:.2f}x fewer quantum executions!")

Solver Comparison on 8-Qubit Spin Glass:
  • BF-DCQO      : 4 circuit evaluations  | Best Energy = -23.1480
  • CD-WarmQAOA  : 35 circuit evaluations | Best Energy = -23.1480
  --> Speedup in QPU Circuit Calls: 8.75x fewer quantum executions!
